# TMC-PINN Experiments

Run cells top to bottom. Each PDE section is independent after setup.

| PDE | Epochs | Switch @ | Est. time (H100) | Cell |
|---|---|---|---|---|
| Reaction | 2,000 | 1,000 | ~15–20 min | 5 |
| Wave | 10,000 | 5,000 | ~2–4 hrs | 6 |
| Allen-Cahn | 10,000 | 5,000 | ~2–4 hrs | 7 |
| Convection | 50,000 | 25,000 | ~8–15 hrs | 8 |
| Burgers' | 10,000 | 5,000 | ~4–6 hrs | 9 |

**Compute budget:** ~$3.29/hr on H100 PCIe. Confirm Lambda balance before starting long runs.

In [ ]:
# Cell 1 — GPU check
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM    : {props.total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — training will be very slow')

In [ ]:
# Cell 2 — Clone / update repo
import os

if os.path.exists('PInnns'):
    os.chdir('PInnns')
    os.system('git pull origin main')
else:
    os.system('git clone https://github.com/michae6345-crypto/PInnns.git')
    os.chdir('PInnns')

print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# Cell 3 — Load train_pinn.py
import importlib.util, sys

if 'train_pinn' in sys.modules:
    del sys.modules['train_pinn']

spec = importlib.util.spec_from_file_location('train_pinn', './train_pinn.py')
tp   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tp)

print('train_pinn.py loaded')
print('Available PDEs:', list(tp.PDE_BUILDERS.keys()))
print('Epoch budgets: ', tp.PDE_EPOCHS)

In [ ]:
# Cell 4 — Smoke test (~1 min, 50 epochs)
# Run this first every session to confirm the environment is working.
tp.main(
    pde          = 'reaction',
    dtype_start  = 'fp64',
    optim_start  = 'adam',
    total_epochs = 50,
    out_dir      = './results_smoke'
)
print('Smoke test passed.')

---
## Main PDE Runs
Each cell runs all 7 conditions for one PDE. `skip_existing=True` means already-completed conditions are skipped safely if the kernel dies mid-run.

In [ ]:
# Cell 5 — REACTION  (~15–20 min total, 2,000 epochs x 7 conditions)
# u_t = rho*u*(1-u), rho=5.  Epoch budget from Xu et al.
tp.run_all(
    pdes          = ['reaction'],
    out_dir       = './results',
    skip_existing = True
)

In [ ]:
# Cell 6 — WAVE  (~2–4 hrs total, 10,000 epochs x 7 conditions)
# u_tt = 4*u_xx.  Epoch budget from Xu et al.
tp.run_all(
    pdes          = ['wave'],
    out_dir       = './results',
    skip_existing = True
)

In [ ]:
# Cell 7 — ALLEN-CAHN  (~2–4 hrs total, 10,000 epochs x 7 conditions)
# u_t - 5u + 5u^3 - 1e-4*u_xx = 0.  IC: x^2*cos(pi*x).
# Loss weights 10:1:1:100 (res:bc1:bc2:ic).  allen_cahn.mat in repo root.
tp.run_all(
    pdes          = ['ac'],
    out_dir       = './results',
    skip_existing = True
)

In [ ]:
# Cell 8 — CONVECTION  (~8–15 hrs total, 50,000 epochs x 7 conditions)
# beta=50, 401x401 grid.  Run last — by far the longest.
# Confirm Lambda balance before starting this cell.
tp.run_all(
    pdes          = ['convection'],
    out_dir       = './results',
    skip_existing = True
)

In [ ]:
# Cell 9 — BURGERS'  (~4–6 hrs total, 10,000 epochs x 7 conditions)
# Formulation: Raissi, Perdikaris & Karniadakis 2019, J. Comput. Phys. 378:686-707
# u_t + u*u_x - (0.01/pi)*u_xx = 0,  x in [-1,1],  t in [0,1]
# IC: u(x,0) = -sin(pi*x)   BC: u(-1,t) = u(1,t) = 0  (Dirichlet)
# Epoch budget is our own choice -- no Xu et al. precedent. Disclose in paper.
# burgers_shock.mat must be in repo root (already committed).
tp.run_all(
    pdes          = ['burgers'],
    out_dir       = './results',
    skip_existing = True
)

---
## Utilities

In [ ]:
# Cell 10 — Check results summary
import os, pandas as pd

CONDITION_LABELS = [
    'fp64lbfgs', 'fp64adam', 'fp32lbfgs', 'fp32adam',
    'fp32adam_to_fp64adam', 'fp32lbfgs_to_fp64lbfgs', 'fp32adam_to_fp64lbfgs'
]
PDES = ['reaction', 'wave', 'ac', 'convection', 'burgers']

out_dir = './results'
missing = []

for pde in PDES:
    for label in CONDITION_LABELS:
        path = os.path.join(out_dir, pde, label, f'{pde}_PINN_{label}_eval.csv')
        if not os.path.exists(path):
            missing.append(f'{pde}/{label}')
        else:
            try:
                df = pd.read_csv(path)
                l2 = df['rel_l2'].iloc[-1] if 'rel_l2' in df.columns else 'N/A'
                print(f'OK  {pde:12s} {label:35s}  L2={l2:.4f}' if l2 != 'N/A' else f'OK  {pde:12s} {label}')
            except Exception as e:
                missing.append(f'{pde}/{label} (read error: {e})')

if missing:
    print(f'\nMISSING ({len(missing)}):')
    for m in missing: print(f'  {m}')
else:
    print('\nAll expected results present.')

In [ ]:
# Cell 11 — Generate paper figures
import importlib.util, os

spec = importlib.util.spec_from_file_location('ag', './aggregate_results.py')
ag   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ag)

ag.main(results_dir='./results', out_dir='./paper_figures')
print('Figures written to ./paper_figures')

In [ ]:
# Cell 12 — Backup to zip (download before closing Lambda)
import zipfile, datetime, os

ts       = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zip_name = f'results_backup_{ts}.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['./results', './paper_figures']:
        if os.path.exists(folder):
            for root, dirs, files in os.walk(folder):
                for f in files:
                    zf.write(os.path.join(root, f))

print(f'Backup written: {zip_name}  ({os.path.getsize(zip_name)/1e6:.1f} MB)')
print('Right-click the file in the Jupyter browser and Download before closing the instance.')